# Corpus Re-embedding + Retrieval Experiments

Replaces MiniLM-L6-v2 (384-dim, general-purpose) with a biomedical embedding model.
Run on Colab A100 or L4 — CPU-only will take hours for 374k docs.

**What this notebook does:**
1. Encodes all 374k docs via `scripts/build_embeddings.py` and saves as `.npy`
2. Evaluates embedding-only retrieval: MiniLM baseline vs new model
3. Evaluates hybrid retrieval: new model + BM25 via Reciprocal Rank Fusion
4. Runs the full pipeline (sim→SVM→SciBERT) with new embeddings
5. Pushes best embeddings to HuggingFace Hub

**Recommended models (set in Config cell):**
- `--medcpt` — `ncbi/MedCPT-Article-Encoder` (768-dim, trained on PubMed click-through; use `ncbi/MedCPT-Query-Encoder` at query time)
- `NeuML/pubmedbert-base-embeddings` — 768-dim, strong biomedical baseline, sentence-transformers compatible
- `BAAI/bge-base-en-v1.5` — 768-dim, top MTEB scores

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — encoding 374k docs will be very slow')

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml rank-bm25 huggingface_hub
!pip install -q sympy==1.13.1  # pin for torch compatibility

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml rank-bm25 huggingface_hub

In [ ]:
!git clone https://github.com/semajyllek/ctmatch.git
REPO = '/content/ctmatch'

In [ ]:
# Needed to push embeddings to HuggingFace Hub
# IMPORTANT: use a WRITE token (huggingface.co/settings/tokens → New token → Role: Write)
# A read-only token will give a 403 error on upload
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Needed to push embeddings to HuggingFace Hub
from huggingface_hub import notebook_login
notebook_login()

# ======== EMBEDDING CONFIG ========
# --- CHOOSE ONE ---

# Option A: MedCPT (recommended — biomedical specialist, asymmetric encoders)
EMBED_CMD    = '--medcpt'
EMB_FILE     = 'doc_embeddings_medcpt.npy'
QUERY_MODEL  = 'ncbi/MedCPT-Query-Encoder'

# Option B: PubMedBERT
# EMBED_CMD    = '--model NeuML/pubmedbert-base-embeddings'
# EMB_FILE     = 'doc_embeddings_pubmedbert-base-embeddings.npy'
# QUERY_MODEL  = 'NeuML/pubmedbert-base-embeddings'

# Option C: BGE base
# EMBED_CMD    = '--model BAAI/bge-base-en-v1.5'
# EMB_FILE     = 'doc_embeddings_bge-base-en-v1.5.npy'
# QUERY_MODEL  = 'BAAI/bge-base-en-v1.5'

BATCH_SIZE = 256  # reduce to 128 if OOM

# ======== PIPELINE CONFIG ========
RUN_ENCODE = False  # set True to re-encode corpus (slow — ~1hr on A100)
PUSH_TO_HF = True  # push embeddings to HuggingFace after encoding

# ======== DATASET CONFIG ========
USE_TREC_2021 = True
USE_TREC_2022 = True
USE_KZ        = True

MAX_TOPICS = 75   # set to None for all topics

# ======== EVAL STEPS ========
RUN_DENSE_EVAL    = True   # Step 3: embedding-only cosine retrieval on judged pool
RUN_HYBRID_EVAL   = True   # Step 4: dense + BM25 via RRF on judged pool
RUN_PIPELINE_EVAL = True   # Step 5: full sim→SVM→SciBERT on judged pool

print(f'Embedding : {EMBED_CMD}')
print(f'Output    : {EMB_FILE}')
print(f'Query     : {QUERY_MODEL}')
print(f'Encode    : {RUN_ENCODE}  Push: {PUSH_TO_HF}')
print(f'Datasets  : TREC2021={USE_TREC_2021}  TREC2022={USE_TREC_2022}  KZ={USE_KZ}')
print(f'Eval steps: dense={RUN_DENSE_EVAL}  hybrid={RUN_HYBRID_EVAL}  pipeline={RUN_PIPELINE_EVAL}')

In [ ]:
# --- CHOOSE ONE ---

# Option A: MedCPT (recommended — biomedical specialist, asymmetric encoders)
EMBED_CMD    = '--medcpt'
EMB_FILE     = 'doc_embeddings_medcpt.npy'
QUERY_MODEL  = 'ncbi/MedCPT-Query-Encoder'

# Option B: PubMedBERT (simpler, sentence-transformers compatible)
# EMBED_CMD    = '--model NeuML/pubmedbert-base-embeddings'
# EMB_FILE     = 'doc_embeddings_pubmedbert-base-embeddings.npy'
# QUERY_MODEL  = 'NeuML/pubmedbert-base-embeddings'

# Option C: BGE base
# EMBED_CMD    = '--model BAAI/bge-base-en-v1.5'
# EMB_FILE     = 'doc_embeddings_bge-base-en-v1.5.npy'
# QUERY_MODEL  = 'BAAI/bge-base-en-v1.5'

BATCH_SIZE = 256  # reduce to 128 if OOM

print(f'Embedding command : {EMBED_CMD}')
print(f'Output file       : {EMB_FILE}')
print(f'Query model       : {QUERY_MODEL}')

if RUN_ENCODE:
    import os
    os.chdir(REPO)
    !python scripts/build_embeddings.py {EMBED_CMD} --output {EMB_FILE} --batch-size {BATCH_SIZE}

In [ ]:
import os
os.chdir(REPO)
!python scripts/build_embeddings.py {EMBED_CMD} --output {EMB_FILE} --batch-size {BATCH_SIZE}

In [ ]:
import numpy as np
new_emb = np.load(f'{REPO}/{EMB_FILE}')
print(f'Shape: {new_emb.shape}  dtype: {new_emb.dtype}  size: {new_emb.nbytes / 1e6:.1f} MB')

import sys
import os
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

sys.path.insert(0, f'{REPO}/src')

from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text,
    calc_first_positive_rank, calc_f1, calc_ndcg,
)

TREC21_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_21_topics.xml')
TREC21_REL_PATH   = os.path.join(DATA_ROOT, 'trec_21_judgments.txt')
TREC22_TOPIC_PATH = os.path.join(DATA_ROOT, 'topics2022.xml')
TREC22_REL_PATH   = os.path.join(DATA_ROOT, 'qrels2022.txt')
KZ_TOPIC_PATH     = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
KZ_REL_PATH       = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

topic2text = {}
rel_dict   = {}

dataset_paths = []
if USE_TREC_2021: dataset_paths.append(('trec', TREC21_TOPIC_PATH, TREC21_REL_PATH))
if USE_TREC_2022: dataset_paths.append(('trec', TREC22_TOPIC_PATH, TREC22_REL_PATH))
if USE_KZ:        dataset_paths.append(('kz',   KZ_TOPIC_PATH,     KZ_REL_PATH))

for kind, topic_path, rel_path in dataset_paths:
    if os.path.exists(topic_path):
        if kind == 'trec':
            topic2text.update(get_trec_topic2text(topic_path))
        else:
            topic2text.update(get_kz_topic2text(topic_path))
    else:
        print(f'WARNING: missing {topic_path}')
    if os.path.exists(rel_path):
        with open(rel_path) as f:
            for line in f:
                tid, _, did, rel = line.split()
                rel_dict.setdefault(tid, {})[did] = int(rel)
    else:
        print(f'WARNING: missing {rel_path}')

HF_ROOT = 'semaj83/ctmatch_ir'
idx_ds = load_dataset(HF_ROOT, data_files='index2docid.txt', split='train')
index2docid = pd.DataFrame(idx_ds)

def get_doc_indices(doc_ids):
    out = []
    for d in doc_ids:
        rows = np.where(index2docid['text'] == d)[0]
        if len(rows): out.append(int(rows[0]))
    return out

print(f'{len(topic2text)} topics loaded, {len(rel_dict)} with judgments')

In [ ]:
import sys
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

sys.path.insert(0, f'{REPO}/src')

from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text,
    calc_first_positive_rank, calc_f1,
)

TREC_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_21_topics.xml')
KZ_TOPIC_PATH   = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
TREC_REL_PATH   = os.path.join(DATA_ROOT, 'trec_21_judgments.txt')
KZ_REL_PATH     = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

topic2text = {}
if os.path.exists(TREC_TOPIC_PATH): topic2text.update(get_trec_topic2text(TREC_TOPIC_PATH))
if os.path.exists(KZ_TOPIC_PATH):   topic2text.update(get_kz_topic2text(KZ_TOPIC_PATH))

rel_dict = {}
for rp in [TREC_REL_PATH, KZ_REL_PATH]:
    if not os.path.exists(rp): continue
    with open(rp) as f:
        for line in f:
            tid, _, did, rel = line.split()
            rel_dict.setdefault(tid, {})[did] = int(rel)

HF_ROOT = 'semaj83/ctmatch_ir'
idx_ds = load_dataset(HF_ROOT, data_files='index2docid.txt', split='train')
index2docid = pd.DataFrame(idx_ds)

def get_doc_indices(doc_ids):
    out = []
    for d in doc_ids:
        rows = np.where(index2docid['text'] == d)[0]
        if len(rows): out.append(int(rows[0]))
    return out

print(f'{len(topic2text)} topics, {len(rel_dict)} with judgments')

## Step 3: Embedding-only retrieval — old vs new
Cosine similarity on judged doc subsets. Gives a clean signal on embedding model quality
before adding SVM/classifier stages.

In [ ]:
# Load old MiniLM embeddings (CSV text format)
print('Loading old MiniLM embeddings from HF...')
old_ds = load_dataset(HF_ROOT, data_files='doc_embeddings.txt', split='train')
old_emb = np.array([np.fromstring(r['text'], sep=',', dtype=np.float32) for r in old_ds])
print(f'Old shape: {old_emb.shape}')

In [ ]:
if RUN_DENSE_EVAL:
    MAX_TOPICS = MAX_TOPICS

    def cosine_rank(topic_emb, emb_matrix, doc_set):
        vecs = emb_matrix[doc_set]
        sims = vecs @ topic_emb / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(topic_emb) + 1e-9)
        return [doc_set[i] for i in np.argsort(-sims)]

    def eval_dense(encode_fn, emb_matrix, name):
        fprs, frrs, f1s, ndcgs = [], [], [], []
        for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
            if topic_id not in rel_dict: continue
            doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
            if not doc_set: continue
            ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)
            ranked_ids = [index2docid.iloc[i]['text'] for i in ranked]
            fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
            fprs.append(fpr); frrs.append(frr)
            f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))
            ndcgs.append(calc_ndcg(ranked_ids, rel_dict[topic_id], k=10))
        return {'Model': name, 'NDCG@10': np.mean(ndcgs), 'MRR': np.mean(frrs), 'F1': np.mean(f1s), 'FPR': np.mean(fprs)}

    old_res = eval_dense(old_encode, old_emb, 'MiniLM-L6-v2 (baseline)')
    new_res = eval_dense(new_encode, new_emb, EMB_FILE)

    import pandas as pd
    pd.DataFrame([old_res, new_res]).set_index('Model').round(4)

In [ ]:
MAX_TOPICS = 75

def cosine_rank(topic_emb, emb_matrix, doc_set):
    vecs = emb_matrix[doc_set]
    sims = vecs @ topic_emb / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(topic_emb) + 1e-9)
    return [doc_set[i] for i in np.argsort(-sims)]

def eval_dense(encode_fn, emb_matrix, name):
    fprs, frrs, f1s = [], [], []
    for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
        if topic_id not in rel_dict: continue
        doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
        if not doc_set: continue
        ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)
        ranked_ids = [index2docid.iloc[i]['text'] for i in ranked]
        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr); f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))
    return {'Model': name, 'FPR↓': np.mean(fprs), 'MRR↑': np.mean(frrs), 'F1↑': np.mean(f1s)}

old_res = eval_dense(old_encode, old_emb, 'MiniLM-L6-v2 (384-dim, baseline)')
new_res = eval_dense(new_encode, new_emb, f'{EMB_FILE}')

pd.DataFrame([old_res, new_res]).set_index('Model').round(4)

## Step 4: Hybrid retrieval — new embeddings + BM25
Reciprocal Rank Fusion combines dense and sparse rankings per topic.
BM25 is built on the judged subset only for efficiency.

In [ ]:
if RUN_HYBRID_EVAL:
    from rank_bm25 import BM25Okapi

    def rrf(rankings, k=60):
        scores = {}
        for ranking in rankings:
            for rank, doc_id in enumerate(ranking):
                scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    def eval_hybrid(encode_fn, emb_matrix, name, rrf_k=60):
        fprs, frrs, f1s, ndcgs = [], [], [], []
        for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
            if topic_id not in rel_dict: continue
            doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
            if not doc_set: continue

            dense_ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)
            subset_texts = [doc_texts[i] for i in doc_set]
            bm25 = BM25Okapi([t.split() for t in subset_texts])
            bm25_order = np.argsort(-bm25.get_scores(topic_text.split()))
            bm25_ranked = [doc_set[i] for i in bm25_order]

            fused = rrf([dense_ranked, bm25_ranked], k=rrf_k)
            ranked_ids = [index2docid.iloc[i]['text'] for i in fused]
            fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
            fprs.append(fpr); frrs.append(frr)
            f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))
            ndcgs.append(calc_ndcg(ranked_ids, rel_dict[topic_id], k=10))
        return {'Model': name, 'NDCG@10': np.mean(ndcgs), 'MRR': np.mean(frrs), 'F1': np.mean(f1s), 'FPR': np.mean(fprs)}

    hybrid_old = eval_hybrid(old_encode, old_emb, 'MiniLM + BM25')
    hybrid_new = eval_hybrid(new_encode, new_emb, f'{EMB_FILE} + BM25')

    rows = []
    if RUN_DENSE_EVAL: rows += [old_res, new_res]
    rows += [hybrid_old, hybrid_new]
    pd.DataFrame(rows).set_index('Model').round(4)

In [ ]:
from rank_bm25 import BM25Okapi

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def eval_hybrid(encode_fn, emb_matrix, name, rrf_k=60):
    fprs, frrs, f1s = [], [], []
    for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
        if topic_id not in rel_dict: continue
        doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
        if not doc_set: continue

        dense_ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)

        subset_texts = [doc_texts[i] for i in doc_set]
        bm25 = BM25Okapi([t.split() for t in subset_texts])
        bm25_order = np.argsort(-bm25.get_scores(topic_text.split()))
        bm25_ranked = [doc_set[i] for i in bm25_order]

        fused = rrf([dense_ranked, bm25_ranked], k=rrf_k)
        ranked_ids = [index2docid.iloc[i]['text'] for i in fused]
        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr); f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))

    return {'Model': name, 'FPR↓': np.mean(fprs), 'MRR↑': np.mean(frrs), 'F1↑': np.mean(f1s)}

hybrid_old = eval_hybrid(old_encode, old_emb, 'MiniLM + BM25 (hybrid)')
hybrid_new = eval_hybrid(new_encode, new_emb, f'{EMB_FILE} + BM25 (hybrid)')

pd.DataFrame([old_res, new_res, hybrid_old, hybrid_new]).set_index('Model').round(4)

if PUSH_TO_HF:
    from huggingface_hub import HfApi
    api = HfApi()
    print(f'Pushing {EMB_FILE} to semaj83/ctmatch_ir ...')
    api.upload_file(
        path_or_fileobj=f'{REPO}/{EMB_FILE}',
        path_in_repo=EMB_FILE,
        repo_id='semaj83/ctmatch_ir',
        repo_type='dataset',
    )
    print('Done.')

In [ ]:
if RUN_PIPELINE_EVAL:
    from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator
    from ctmatch.config import PipeConfig
    from ctmatch.matching.pipeline import CTMatch

    pipe_config = PipeConfig(
        ir_setup=True,
        embedding_file=EMB_FILE,
        embedding_model_checkpoint=QUERY_MODEL,
        filters=['sim', 'svm', 'classifier'],
    )
    ctm = CTMatch(pipe_config)

    trec_topic_paths = []
    if USE_TREC_2021 and os.path.exists(TREC21_TOPIC_PATH): trec_topic_paths.append(TREC21_TOPIC_PATH)
    if USE_TREC_2022 and os.path.exists(TREC22_TOPIC_PATH): trec_topic_paths.append(TREC22_TOPIC_PATH)

    eval_config = EvaluatorConfig(
        rel_paths=[p for flag, p in [(USE_TREC_2021, TREC21_REL_PATH), (USE_TREC_2022, TREC22_REL_PATH), (USE_KZ, KZ_REL_PATH)] if flag],
        trec_topic_path=trec_topic_paths or None,
        kz_topic_path=KZ_TOPIC_PATH if USE_KZ else None,
        max_topics=MAX_TOPICS,
        filters=['sim', 'svm', 'classifier'],
    )

    evaluator = Evaluator(eval_config)
    evaluator.ctm = ctm  # inject CTMatch with new embeddings
    output_file = f'eval_predictions_{EMB_FILE.replace(\".npy\", \"\")}.jsonl'
    results = evaluator.evaluate_detailed(output_file)
    print(results)

In [ ]:
from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator
from ctmatch.config import PipeConfig
from ctmatch.matching.pipeline import CTMatch

pipe_config = PipeConfig(
    ir_setup=True,
    embedding_file=EMB_FILE,
    embedding_model_checkpoint=QUERY_MODEL,
    filters=['sim', 'svm', 'classifier'],
)
ctm = CTMatch(pipe_config)

eval_config = EvaluatorConfig(
    rel_paths=[TREC_REL_PATH, KZ_REL_PATH],
    trec_topic_path=TREC_TOPIC_PATH,
    kz_topic_path=KZ_TOPIC_PATH,
    max_topics=MAX_TOPICS,
    filters=['sim', 'svm', 'classifier'],
)

evaluator = Evaluator(eval_config)
evaluator.ctm = ctm  # inject our pre-loaded CTMatch with new embeddings
results = evaluator.evaluate_detailed(f'eval_predictions_{EMB_FILE.replace(".npy", "")}.jsonl')
print(results)